In [1]:
# Задание 1

import math

alpha = 3.0       # среднее время обслуживания (часы)
n = 24            # заявок в сутки
k = 4             # число каналов

# Интенсивность поступления заявок (в час)
lam = n / 24
# Интенсивность обслуживания
mu = 1 / alpha
# Приведённая нагрузка
rho = lam / mu

print(f"Интенсивность поступления (lambda): {lam:.3f} з/час")
print(f"Интенсивность обслуживания (mu): {mu:.3f} з/час")
print(f"Приведённая нагрузка (rho): {rho:.3f}\n")

# 1. Поиск минимального числа каналов для Q >= 0.95
for channels in range(1, 20):
    denominator = sum((rho ** i) / math.factorial(i) for i in range(channels + 1))
    p0 = 1 / denominator
    p_refusal = ((rho ** channels) / math.factorial(channels)) * p0
    Q = 1 - p_refusal
    if Q >= 0.95:
        print(f"Минимальное число каналов (Q >= 0.95): k = {channels} (Q = {Q:.3f})")
        break

# 2-3. Расчёты для заданного k = 4
denominator = sum((rho ** i) / math.factorial(i) for i in range(k + 1))
p0 = 1 / denominator

probabilities = [((rho ** i) / math.factorial(i)) * p0 for i in range(k + 1)]

p_refusal = probabilities[k]
Q = 1 - p_refusal
A = lam * Q
busy_channels = rho * Q
load_factor = busy_channels / k

print(f"\n--- Характеристики для k = 4 ---")
print(f"Вероятность отказа: {p_refusal:.3f}")
print(f"Относительная пропускная способность (Q): {Q:.3f}")
print(f"Абсолютная пропускная способность (A): {A:.3f} з/час")
print(f"Среднее число занятых каналов: {busy_channels:.3f}")
print(f"Коэффициент загрузки каналов: {load_factor:.3f}")

Интенсивность поступления (lambda): 1.000 з/час
Интенсивность обслуживания (mu): 0.333 з/час
Приведённая нагрузка (rho): 3.000

Минимальное число каналов (Q >= 0.95): k = 7 (Q = 0.978)

--- Характеристики для k = 4 ---
Вероятность отказа: 0.206
Относительная пропускная способность (Q): 0.794
Абсолютная пропускная способность (A): 0.794 з/час
Среднее число занятых каналов: 2.382
Коэффициент загрузки каналов: 0.595


Система с 4 каналами не справляется с заданным критерием качества (относительная пропускная способность составляет лишь 79.4% вместо требуемых 95%). Из-за высокой загрузки (ρ=3) более 20% заявок получают отказ. Для обеспечения пропускной способности не менее 95% необходимо увеличить количество каналов до 7.

In [2]:
# Задание 2
import math

lam = 120 / 24 # з/час
t = 8          # минут
alpha_cost = 4
n_limit = 2

mu = 60 / t    # з/час
rho = lam / mu

# Минимальное число каналов (условие стационарности rho/k < 1)
kmin = math.floor(rho) + 1

def calculate_smo(k):
    sum1 = sum((rho ** i) / math.factorial(i) for i in range(k))
    sum2 = ((rho ** k) / math.factorial(k)) * (1 / (1 - rho / k))
    p0 = 1 / (sum1 + sum2)
    
    # ИСПРАВЛЕНИЕ: Добавлен k в знаменатель!
    Lq = ((rho ** (k + 1)) / (k * math.factorial(k) * ((1 - rho / k) ** 2))) * p0
    L = Lq + rho
    Wq = Lq / lam
    W = L / lam
    # Считаем издержки по времени пребывания
    C = (k / lam) + alpha_cost * W 
    return p0, Lq, L, Wq, W, C

print(f"kmin = {kmin} (rho = {rho:.3f})")
p0_min, Lq_min, L_min, Wq_min, W_min, C_min = calculate_smo(kmin)

print(f"\n--- Для kmin = {kmin} ---")
print(f"Очередь (Lq): {Lq_min:.3f}, В системе (L): {L_min:.3f}")
print(f"Время ожидания (Wq): {Wq_min:.3f} ч, Пребывания (W): {W_min:.3f} ч")
print(f"Затраты (C): {C_min:.3f}")

# Оптимальное число каналов
best_k, best_cost = kmin, C_min
for k in range(kmin + 1, 10):
    _, _, _, _, _, C_k = calculate_smo(k)
    if C_k < best_cost:
        best_cost = C_k
        best_k = k

print(f"\nОптимальное количество каналов kopt = {best_k}")
p0_opt, Lq_opt, L_opt, Wq_opt, W_opt, C_opt = calculate_smo(best_k)

print(f"--- Для kopt = {best_k} ---")
print(f"Очередь (Lq): {Lq_opt:.3f}, В системе (L): {L_opt:.3f}")
print(f"Время ожидания (Wq): {Wq_opt:.3f} ч, Пребывания (W): {W_opt:.3f} ч")
print(f"Затраты (C): {C_opt:.3f}")

# Вероятность, что в очереди <= n заявок
p0 = p0_opt
prob_queue_le_n = 0
for i in range(best_k + n_limit + 1):
    if i < best_k:
        pi = ((rho ** i) / math.factorial(i)) * p0
    else:
        pi = ((rho ** i) / (math.factorial(best_k) * (best_k ** (i - best_k)))) * p0
    prob_queue_le_n += pi

print(f"\nВероятность P(очередь <= {n_limit}) при kopt: {prob_queue_le_n:.3f}")

kmin = 1 (rho = 0.667)

--- Для kmin = 1 ---
Очередь (Lq): 1.333, В системе (L): 2.000
Время ожидания (Wq): 0.267 ч, Пребывания (W): 0.400 ч
Затраты (C): 1.800

Оптимальное количество каналов kopt = 2
--- Для kopt = 2 ---
Очередь (Lq): 0.083, В системе (L): 0.750
Время ожидания (Wq): 0.017 ч, Пребывания (W): 0.150 ч
Затраты (C): 1.000

Вероятность P(очередь <= 2) при kopt: 0.994


Минимально необходимое число каналов для стабильной работы СМО равно 1. Однако при одном канале система работает с существенной очередью (в среднем 1.3 заявки) и большими издержками на ожидание. Оптимизация функции затрат показала, что выгоднее содержать 2 канала обслуживания. Это резко снижает длину очереди (до 0.08) и время пребывания, уменьшая суммарные издержки до минимума. Вероятность того, что в очереди будет не более двух человек при 2 каналах, стремится к 100%.

In [3]:
# Задание 3
import math

lam = 5
t = 1
k = 20
n = 2
T = 8
C = 120

mu = 60 / t
rho = lam / mu

sum1 = sum((rho ** i) / math.factorial(i) for i in range(k + 1))
sum2 = sum(((rho ** (k + i)) / (math.factorial(k) * (k ** i))) for i in range(1, n + 1))

p0 = 1 / (sum1 + sum2)

probabilities = []
for i in range(k + n + 1):
    if i <= k:
        pi = ((rho ** i) / math.factorial(i)) * p0
    else:
        pi = ((rho ** i) / (math.factorial(k) * (k ** (i - k)))) * p0
    probabilities.append(pi)

p_refusal = probabilities[-1]
Q = 1 - p_refusal
A = lam * Q

Lq = sum(i * probabilities[k + i] for i in range(1, n + 1))
busy = A / mu
L = Lq + busy
Wq = Lq / A if A > 0 else 0
Ws = 1 / mu
W = Wq + Ws

loss = C * lam * p_refusal * T

print(f"Приведённая нагрузка (rho): {rho:.3f}")
print(f"P0: {p0:.5f}")
print(f"Вероятность отказа: {p_refusal:.10f}")
print(f"Пропускная способность: абс. = {A:.3f}, отн. = {Q:.3f}")
print(f"Lq (в очереди): {Lq:.5f}, L (в системе): {L:.3f}")
print(f"Wq (ожидание): {Wq:.5f} ч, W (пребывание): {W:.3f} ч")
print(f"Потери выручки: {loss:.5f} у.е.")

Приведённая нагрузка (rho): 0.083
P0: 0.92004
Вероятность отказа: 0.0000000000
Пропускная способность: абс. = 5.000, отн. = 1.000
Lq (в очереди): 0.00000, L (в системе): 0.083
Wq (ожидание): 0.00000 ч, W (пребывание): 0.017 ч
Потери выручки: 0.00000 у.е.


Результаты показывают абсолютную избыточность выделенных ресурсов для СМО. Имея 20 каналов при интенсивности потока 5 заявок в час и высокой скорости обслуживания (1 мин/заявка), система практически всегда простаивает (92% времени система полностью пуста). Отказы в обслуживании и очередь физически отсутствуют, следовательно, финансовые потери из-за отказов равны нулю. Эффективность можно повысить за счёт радикального сокращения числа каналов.

In [4]:
# Задание 4
lam = 0.3       # з/мин
t = 5           # мин
k = 1
omega = 15      # макс ожидание
C_val = 250
eps = 0.01

mu = 1 / t
nu = 1 / omega

# ИСПРАВЛЕНИЕ: Динамический поиск предела ряда с учетом точности eps
coeffs = [1.0]
i = 1
while True:
    if i <= k:
        c = coeffs[i - 1] * lam / (i * mu)
    else:
        c = coeffs[i - 1] * lam / (k * mu + (i - k) * nu)
    
    coeffs.append(c)
    
    # Оценка точности
    current_p0 = 1 / sum(coeffs)
    current_pi = c * current_p0
    if i > k and current_pi < eps:
        break
    i += 1

p0 = 1 / sum(coeffs)
probs = [c * p0 for c in coeffs]

L = sum(j * probs[j] for j in range(len(probs)))
Lq = sum((j - k) * probs[j] for j in range(k + 1, len(probs)))

# ИСПРАВЛЕНИЕ: Корректный расчёт P_обсл
nu_leave = nu * Lq
P_service = (lam - nu_leave) / lam

W = L / lam
Wq = Lq / lam
loss = C_val * nu_leave

print(f"Количество учтённых состояний для точности {eps}: {len(probs)}")
print(f"Вероятность обслуживания: {P_service:.3f}")
print(f"Среднее число заявок: в очереди (Lq) = {Lq:.3f}, в системе (L) = {L:.3f}")
print(f"Среднее время: ожидания (Wq) = {Wq:.3f} мин, пребывания (W) = {W:.3f} мин")
print(f"Интенсивность ухода: {nu_leave:.3f} з/мин")
print(f"Средние потери дохода: {loss:.3f} у.е./мин")

Количество учтённых состояний для точности 0.01: 10
Вероятность обслуживания: 0.591
Среднее число заявок: в очереди (Lq) = 1.840, в системе (L) = 2.720
Среднее время: ожидания (Wq) = 6.133 мин, пребывания (W) = 9.066 мин
Интенсивность ухода: 0.123 з/мин
Средние потери дохода: 30.664 у.е./мин


Система сильно перегружена: интенсивность поступления заявок превышает интенсивность обслуживания (λ=0.3>μ=0.2). Бесконечного роста очереди не происходит исключительно из-за ухода нетерпеливых клиентов по истечении 15 минут. Система успешно обслуживает лишь около 60% потока, остальные 40% теряются, что приводит к значительным финансовым издержкам (около 30 у.е. в минуту).

In [5]:
# Задание 5
import math

n = 16
k = 5
t = 1.0
P_required = 98

lam = k / 30   # интенсивность одного источника в день
mu = 1 / t     # интенсивность ремонта в день

# Формулы Энгсета для одноканальной замкнутой СМО
denominator = sum((math.factorial(n) / math.factorial(n - i)) * ((lam / mu) ** i) for i in range(n + 1))
p0 = 1 / denominator

probs = [((math.factorial(n) / math.factorial(n - i)) * ((lam / mu) ** i) * p0) for i in range(n + 1)]

# Ищем вероятность того, что исправно >= P_required %
threshold = math.ceil(n * P_required / 100) # Нужно 16 исправных
# 16 исправных значит 0 в ремонте (состояние P0)
prob_active = sum(probs[i] for i in range(0, n - threshold + 1))

L = sum(i * probs[i] for i in range(n + 1))
A = mu * (1 - p0)
W = L / A if A > 0 else 0

print(f"Вероятность того, что исправно >= {P_required}% ({threshold} шт.): {prob_active:.5f}")
print(f"Среднее число неисправных: {L:.3f}")
print(f"Абсолютная пропускная способность: {A:.3f} з/день")
print(f"Среднее время в ремонте (и ожидании): {W:.3f} дней")

Вероятность того, что исправно >= 98% (16 шт.): 0.00033
Среднее число неисправных: 10.002
Абсолютная пропускная способность: 1.000 з/день
Среднее время в ремонте (и ожидании): 10.005 дней


СМО критически не справляется с поддержанием оборудования в рабочем состоянии. Вероятность того, что 98% источников исправны, стремится к нулю. В среднем из 16 аппаратов более 10 одновременно находятся в ремонте или ожидают его. Это обусловлено тем, что один канал может чинить лишь 1 заявку в день, в то время как общий поток заявок значительно превосходит его возможности. Необходимо увеличить количество ремонтных бригад.

In [6]:
# Задание 6
import math

k = 4
n = 16
lam = 1.5
t = 0.1

mu = 1 / t

coeffs = []
for i in range(n + 1):
    numerator = math.factorial(n)
    denominator = math.factorial(n - i)
    prod = 1
    for j in range(1, i + 1):
        prod *= min(j, k) * mu
    c = (numerator / denominator) * ((lam ** i) / prod)
    coeffs.append(c)

p0 = 1 / sum(coeffs)
probs = [p0 * c for c in coeffs]

L = sum(i * probs[i] for i in range(n + 1))
busy = sum(min(i, k) * probs[i] for i in range(n + 1))
free = k - busy
Lq = L - busy

A = mu * busy
# ИСПРАВЛЕНИЕ: Q = A / (реальный поток заявок)
real_lambda = lam * (n - L)
Q = A / real_lambda if real_lambda > 0 else 0

queue_prob = sum(probs[i] for i in range(k + 1, n + 1))

Wq = Lq / A if A > 0 else 0
Ws = 1 / mu
W = Wq + Ws

print(f"Вероятность отсутствия заявок P0 (все свободны): {p0:.3f}")
print(f"Среднее число заявок: в очереди Lq = {Lq:.3f}, в системе L = {L:.3f}")
print(f"Каналы: занято = {busy:.3f}, свободно = {free:.3f}")
print(f"Пропускная способность: абс. = {A:.3f}, отн. = {Q:.3f}")
print(f"Вероятность наличия очереди: {queue_prob:.3f}")
print(f"Время: ожидания Wq = {Wq:.3f} ч, пребывания W = {W:.3f} ч")

Вероятность отсутствия заявок P0 (все свободны): 0.104
Среднее число заявок: в очереди Lq = 0.113, в системе L = 2.185
Каналы: занято = 2.072, свободно = 1.928
Пропускная способность: абс. = 20.722, отн. = 1.000
Вероятность наличия очереди: 0.071
Время: ожидания Wq = 0.005 ч, пребывания W = 0.105 ч


Замкнутая система работает в высокоэффективном, стабильном режиме. Каналы обслуживания загружены оптимально (в среднем занята половина: ~2 из 4). Благодаря быстрой обработке (t=0.1 ч) очередь образуется крайне редко (вероятность 7.1%), а среднее время ожидания пренебрежимо мало. СМО полностью справляется со входящим потоком от источников без образования "узких мест".